In [1]:
# ==============================================================
# CELL 1: THƯ VIỆN, ĐƯỜNG DẪN VÀ THAM SỐ
# ==============================================================
import os
import numpy as np
import pandas as pd
import zipfile

BASE_DIR = os.getcwd()

RAW_FILENAME = "flights.csv"
ZIP_FILENAME = "flights.zip"
AIRPORTS_FILENAME = "airports.csv"
BTS_FILENAME = "T_MASTER_CORD.csv"

AIRLINE_CODE = "DL"  # Delta Air Lines
FILTERED_RAW_FILENAME = f"flights_{AIRLINE_CODE}_raw.csv"
CLEANED_FILENAME = f"flights_{AIRLINE_CODE}_cleaned.csv"

RAW_PATH = os.path.join(BASE_DIR, RAW_FILENAME)
ZIP_PATH = os.path.join(BASE_DIR, ZIP_FILENAME)
AIRPORTS_PATH = os.path.join(BASE_DIR, AIRPORTS_FILENAME)
BTS_PATH = os.path.join(BASE_DIR, BTS_FILENAME)
FILTERED_RAW_PATH = os.path.join(BASE_DIR, FILTERED_RAW_FILENAME)
CLEANED_PATH = os.path.join(BASE_DIR, CLEANED_FILENAME)


In [2]:
# ==============================================================
# CELL 2: GIẢI NÉN, ĐỌC VÀ LỌC DELTA NGAY TỪ DỮ LIỆU GỐC
# ==============================================================
# Nếu chỉ có flights.zip, tự động giải nén flights.csv
if not os.path.exists(RAW_PATH):
    if not os.path.exists(ZIP_PATH):
        raise FileNotFoundError("Cần đặt flights.csv hoặc flights.zip trong thư mục làm việc.")
    with zipfile.ZipFile(ZIP_PATH, "r") as z:
        z.extractall(BASE_DIR)
    print(f"[1] Đã giải nén {ZIP_FILENAME}.")

airports = pd.read_csv(AIRPORTS_PATH, dtype=str)
VALID_AIRPORT_CODES = set(airports["IATA_CODE"].dropna().str.strip().str.upper())

# Lọc theo chunk để tránh nạp toàn bộ 5,8 triệu dòng vào RAM
CHUNK_SIZE = 500_000
chunks = []
total_rows = 0
for chunk in pd.read_csv(
    RAW_PATH,
    chunksize=CHUNK_SIZE,
    dtype={"ORIGIN_AIRPORT": str, "DESTINATION_AIRPORT": str},
    low_memory=False
):
    total_rows += len(chunk)
    selected = chunk.loc[chunk["AIRLINE"].eq(AIRLINE_CODE)].copy()
    if not selected.empty:
        chunks.append(selected)

df = pd.concat(chunks, ignore_index=True)
print(f"[1] Dữ liệu gốc: {total_rows:,} dòng.")
print(f"[1b] Sau khi lọc hãng {AIRLINE_CODE}: {len(df):,} dòng, {df.shape[1]} cột.")
df.to_csv(FILTERED_RAW_PATH, index=False)
print(f"[1c] Đã lưu dữ liệu Delta chưa làm sạch: {FILTERED_RAW_FILENAME}")


[1] Dữ liệu gốc: 5,819,079 dòng.
[1b] Sau khi lọc hãng DL: 875,881 dòng, 31 cột.
[1c] Đã lưu dữ liệu Delta chưa làm sạch: flights_DL_raw.csv


In [3]:
# ==============================================================
# CELL 3: TẠO CỘT NGÀY ĐẦY ĐỦ (FLIGHT_DATE)
# ==============================================================
df["FLIGHT_DATE"] = pd.to_datetime(
    dict(year=df["YEAR"], month=df["MONTH"], day=df["DAY"])
)
print(
    f"[2] Đã tạo FLIGHT_DATE cho dữ liệu Delta. "
    f"Ngày đầu tiên: {df['FLIGHT_DATE'].iloc[0].date()}"
)


[2] Đã tạo FLIGHT_DATE cho dữ liệu Delta. Ngày đầu tiên: 2015-01-01


In [4]:
# ============================================================== 
# CELL 4: GIẢI MÃ VÀ CHUẨN HÓA MÃ SÂN BAY TRÊN DỮ LIỆU DELTA
# ============================================================== 
bts = pd.read_csv(BTS_PATH, dtype=str)
bts_latest = (
    bts[["AIRPORT_ID", "AIRPORT"]]
    .dropna(subset=["AIRPORT_ID", "AIRPORT"])
    .drop_duplicates(subset="AIRPORT_ID", keep="last")
)
ID_TO_IATA = dict(zip(bts_latest["AIRPORT_ID"], bts_latest["AIRPORT"]))
print(
    f"[3] Đã đọc bảng tra cứu BTS: {len(ID_TO_IATA):,} mã AIRPORT_ID -> IATA."
)


def resolve_airport_code(code: str) -> str:
    """Tra cứu chuyển đổi mã DOT ID dạng số sang mã IATA chuẩn."""
    code = str(code).strip().upper()
    if code in VALID_AIRPORT_CODES:
        return code
    return ID_TO_IATA.get(code, code)


origin_before = df["ORIGIN_AIRPORT"].astype(str).str.strip().str.upper()
dest_before = df["DESTINATION_AIRPORT"].astype(str).str.strip().str.upper()

df["ORIGIN_AIRPORT"] = origin_before.apply(resolve_airport_code)
df["DESTINATION_AIRPORT"] = dest_before.apply(resolve_airport_code)

n_resolved_origin = (origin_before != df["ORIGIN_AIRPORT"]).sum()
n_resolved_dest = (dest_before != df["DESTINATION_AIRPORT"]).sum()
print(
    f"[3] Đã ánh xạ lại (số -> IATA) cho {n_resolved_origin:,} dòng ở ORIGIN, "
    f"{n_resolved_dest:,} dòng ở DESTINATION nhờ bảng BTS."
)

# Flag các mã lạ thực sự không tồn tại ở cả 2 bảng tra cứu
df["ORIGIN_AIRPORT_VALID"] = (
    df["ORIGIN_AIRPORT"].isin(VALID_AIRPORT_CODES).astype(int)
)
df["DEST_AIRPORT_VALID"] = (
    df["DESTINATION_AIRPORT"].isin(VALID_AIRPORT_CODES).astype(int)
)

df["ORIGIN_AIRPORT_CLEAN"] = np.where(
    df["ORIGIN_AIRPORT_VALID"], df["ORIGIN_AIRPORT"], "UNKNOWN"
)
df["DEST_AIRPORT_CLEAN"] = np.where(
    df["DEST_AIRPORT_VALID"], df["DESTINATION_AIRPORT"], "UNKNOWN"
)

n_invalid_origin = (df["ORIGIN_AIRPORT_VALID"] == 0).sum()
n_invalid_dest = (df["DEST_AIRPORT_VALID"] == 0).sum()
print(
    f"[3b] Các mã lạ KHÔNG thể tra cứu: {n_invalid_origin} ở ORIGIN, {n_invalid_dest} ở DESTINATION -> Đánh dấu UNKNOWN."
)

[3] Đã đọc bảng tra cứu BTS: 6,903 mã AIRPORT_ID -> IATA.
[3] Đã ánh xạ lại (số -> IATA) cho 75,552 dòng ở ORIGIN, 75,552 dòng ở DESTINATION nhờ bảng BTS.
[3b] Các mã lạ KHÔNG thể tra cứu: 332 ở ORIGIN, 331 ở DESTINATION -> Đánh dấu UNKNOWN.


In [5]:
# ==============================================================
# CELL 5: CHUẨN HÓA DỮ LIỆU LOGIC VÀ XỬ LÝ MISSING VALUES
# ==============================================================
# 1. Chuyển CANCELLED/DIVERTED về kiểu nhị phân (0/1)
df["CANCELLED"] = df["CANCELLED"].astype(int)
df["DIVERTED"] = df["DIVERTED"].astype(int)

# 2. Xử lý thiếu ở các cột lý do trễ cho các chuyến bay HOÀN THÀNH
delay_reason_cols = [
    "AIR_SYSTEM_DELAY",
    "SECURITY_DELAY",
    "AIRLINE_DELAY",
    "LATE_AIRCRAFT_DELAY",
    "WEATHER_DELAY",
]
mask_completed = df["CANCELLED"] == 0
for c in delay_reason_cols:
    df.loc[mask_completed, c] = df.loc[mask_completed, c].fillna(0)

# 3. Điền giá trị mặc định cho phân loại bị hủy/số hiệu đuôi bay
df["CANCELLATION_REASON"] = df["CANCELLATION_REASON"].fillna("NONE")
df["TAIL_NUMBER"] = df["TAIL_NUMBER"].fillna("UNKNOWN")

# 4. Vá lỗi SCHEDULED_TIME bị thiếu (nếu có)
def hhmm_to_minutes(x):
    if pd.isna(x):
        return np.nan
    x = int(x)
    if x == 2400:
        x = 0
    hh, mm = divmod(x, 100)
    return hh * 60 + mm


n_missing_sched_time = df["SCHEDULED_TIME"].isna().sum()
if n_missing_sched_time > 0:
    mask_missing = df["SCHEDULED_TIME"].isna()
    dep_min = df.loc[mask_missing, "SCHEDULED_DEPARTURE"].apply(hhmm_to_minutes)
    arr_min = df.loc[mask_missing, "SCHEDULED_ARRIVAL"].apply(hhmm_to_minutes)
    df.loc[mask_missing, "SCHEDULED_TIME"] = (arr_min - dep_min) % 1440
print(f"[4] Đã vá {n_missing_sched_time} dòng thiếu SCHEDULED_TIME.")

[4] Đã vá 0 dòng thiếu SCHEDULED_TIME.


In [6]:
# ==============================================================
# CELL 6: TẠO BIẾN TARGET & LƯU FILE DELTA ĐÃ LÀM SẠCH
# ==============================================================
# Tạo nhãn mục tiêu IS_DELAYED (Trễ từ 15 phút trở lên)
df["IS_DELAYED"] = np.where(
    df["ARRIVAL_DELAY"].isna(), np.nan, (df["ARRIVAL_DELAY"] >= 15).astype(float)
)

# Cảnh báo Data Leakage
LEAKAGE_COLS = [
    "DEPARTURE_TIME",
    "DEPARTURE_DELAY",
    "TAXI_OUT",
    "WHEELS_OFF",
    "ELAPSED_TIME",
    "AIR_TIME",
    "WHEELS_ON",
    "TAXI_IN",
    "ARRIVAL_TIME",
]
print(
    f"[!] CẢNH BÁO DATA LEAKAGE: Không sử dụng các cột {LEAKAGE_COLS} khi huấn luyện mô hình dự đoán trễ chuyến trước giờ bay!"
)

# Lưu dữ liệu đã làm sạch tổng thể
df.to_csv(CLEANED_PATH, index=False)
print(
    f"[5] Đã lưu file làm sạch thành công: {CLEANED_FILENAME} (Shape: {df.shape})"
)

[!] CẢNH BÁO DATA LEAKAGE: Không sử dụng các cột ['DEPARTURE_TIME', 'DEPARTURE_DELAY', 'TAXI_OUT', 'WHEELS_OFF', 'ELAPSED_TIME', 'AIR_TIME', 'WHEELS_ON', 'TAXI_IN', 'ARRIVAL_TIME'] khi huấn luyện mô hình dự đoán trễ chuyến trước giờ bay!
[5] Đã lưu file làm sạch thành công: flights_DL_cleaned.csv (Shape: (875881, 37))


In [7]:
# ==============================================================
# CELL 7: KIỂM TRA VÀ LƯU DỮ LIỆU DELTA ĐÃ LÀM SẠCH
# ==============================================================
# Kiểm tra phạm vi dữ liệu sau toàn bộ pipeline
assert df["AIRLINE"].eq(AIRLINE_CODE).all(), "Phát hiện dòng không thuộc Delta!"

print(f"[6] Kiểm tra AIRLINE: {df['AIRLINE'].unique().tolist()}")
print(f"[6] Kích thước dữ liệu Delta sau làm sạch: {df.shape}")

# Lưu dữ liệu Delta đã làm sạch
df.to_csv(CLEANED_PATH, index=False)
print(f"[6] Đã lưu file: {CLEANED_FILENAME}")


[6] Kiểm tra AIRLINE: ['DL']
[6] Kích thước dữ liệu Delta sau làm sạch: (875881, 37)
[6] Đã lưu file: flights_DL_cleaned.csv


In [9]:
# ============================================================
# KIỂM TRA TỔNG QUAN DỮ LIỆU DELTA SAU KHI LÀM SẠCH
# ============================================================

import pandas as pd
import numpy as np

# Đọc dữ liệu đã làm sạch
df = pd.read_csv(
    "flights_DL_cleaned.csv",
    low_memory=False
)

print("=" * 60)
print("1. TỔNG QUAN DỮ LIỆU")
print("=" * 60)

print(f"Số dòng: {df.shape[0]:,}")
print(f"Số cột: {df.shape[1]:,}")
print(f"Kích thước dữ liệu: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

if "AIRLINE" in df.columns:
    print(f"Hãng hàng không: {df['AIRLINE'].unique()}")

display(df.head())


# ============================================================
# 2. KIỂM TRA KIỂU DỮ LIỆU
# ============================================================

print("=" * 60)
print("2. KIỂU DỮ LIỆU")
print("=" * 60)

display(
    pd.DataFrame({
        "Data_Type": df.dtypes,
        "Non_Null": df.notna().sum(),
        "Unique_Values": df.nunique()
    })
)


# ============================================================
# 3. KIỂM TRA MISSING VALUES
# ============================================================

print("=" * 60)
print("3. MISSING VALUES")
print("=" * 60)

missing_summary = pd.DataFrame({
    "Missing_Count": df.isna().sum(),
    "Missing_Percentage": (
        df.isna().mean() * 100
    ).round(2)
})

# Chỉ hiển thị các cột có giá trị thiếu
missing_summary = missing_summary[
    missing_summary["Missing_Count"] > 0
].sort_values(
    by="Missing_Count",
    ascending=False
)

if missing_summary.empty:
    print("Không có giá trị thiếu trong dữ liệu.")
else:
    display(missing_summary)

print(
    f"Tổng số giá trị thiếu: "
    f"{df.isna().sum().sum():,}"
)


# ============================================================
# 4. KIỂM TRA IS_DELAYED
# ============================================================

print("=" * 60)
print("4. PHÂN PHỐI IS_DELAYED")
print("=" * 60)

if "IS_DELAYED" in df.columns:

    # Số lượng từng nhóm
    delayed_count = df["IS_DELAYED"].value_counts(
        dropna=False
    )

    print("Số lượng từng nhóm:")
    display(delayed_count.to_frame("Count"))

    # Tỷ lệ phần trăm từng nhóm
    delayed_percentage = (
        df["IS_DELAYED"]
        .value_counts(normalize=True, dropna=False) * 100
    ).round(2)

    print("Tỷ lệ phần trăm từng nhóm:")
    display(delayed_percentage.to_frame("Percentage"))

    # Tổng hợp số chuyến bị trễ
    delayed_flights = (df["IS_DELAYED"] == 1).sum()
    on_time_flights = (df["IS_DELAYED"] == 0).sum()

    print(f"Số chuyến bị trễ: {delayed_flights:,}")
    print(f"Số chuyến không bị trễ: {on_time_flights:,}")

else:
    print(
        "Không tìm thấy cột IS_DELAYED. "
        "Hãy kiểm tra lại bước tạo biến."
    )


# ============================================================
# 5. THỐNG KÊ CƠ BẢN CÁC BIẾN SỐ
# ============================================================

print("=" * 60)
print("5. THỐNG KÊ CƠ BẢN")
print("=" * 60)

numeric_columns = df.select_dtypes(
    include=np.number
).columns

display(
    df[numeric_columns].describe().T
)


# ============================================================
# 6. KIỂM TRA DỮ LIỆU TRÙNG LẶP
# ============================================================

print("=" * 60)
print("6. DỮ LIỆU TRÙNG LẶP")
print("=" * 60)

duplicate_count = df.duplicated().sum()

print(f"Số dòng trùng lặp: {duplicate_count:,}")

if duplicate_count == 0:
    print("Không phát hiện dòng trùng lặp hoàn toàn.")


# ============================================================
# 7. TỔNG KẾT
# ============================================================

print("=" * 60)
print("7. TỔNG KẾT")
print("=" * 60)

print(f"- Số dòng: {len(df):,}")
print(f"- Số cột: {len(df.columns):,}")
print(f"- Tổng giá trị thiếu: {df.isna().sum().sum():,}")
print(f"- Số dòng trùng lặp: {df.duplicated().sum():,}")

if "IS_DELAYED" in df.columns:
    print(
        f"- Tỷ lệ chuyến bị trễ: "
        f"{(df['IS_DELAYED'] == 1).mean() * 100:.2f}%"
    )

print("Hoàn thành kiểm tra dữ liệu Delta.")

1. TỔNG QUAN DỮ LIỆU
Số dòng: 875,881
Số cột: 37
Kích thước dữ liệu: 549.61 MB
Hãng hàng không: <StringArray>
['DL']
Length: 1, dtype: str


,YEAR,MONTH,DAY,DAY_OF_WEEK,AIRLINE,FLIGHT_NUMBER,TAIL_NUMBER,ORIGIN_AIRPORT,DESTINATION_AIRPORT,SCHEDULED_DEPARTURE,...,SECURITY_DELAY,AIRLINE_DELAY,LATE_AIRCRAFT_DELAY,WEATHER_DELAY,FLIGHT_DATE,ORIGIN_AIRPORT_VALID,DEST_AIRPORT_VALID,ORIGIN_AIRPORT_CLEAN,DEST_AIRPORT_CLEAN,IS_DELAYED
0,2015,1,1,4,DL,806,N3730B,SFO,MSP,25,...,0.0,0.0,0.0,0.0,2015-01-01,1,1,SFO,MSP,0.0
1,2015,1,1,4,DL,1173,N826DN,LAS,ATL,30,...,0.0,0.0,0.0,0.0,2015-01-01,1,1,LAS,ATL,0.0
2,2015,1,1,4,DL,2336,N958DN,DEN,ATL,30,...,0.0,0.0,0.0,0.0,2015-01-01,1,1,DEN,ATL,0.0
3,2015,1,1,4,DL,1434,N547US,LAX,MSP,35,...,0.0,0.0,0.0,0.0,2015-01-01,1,1,LAX,MSP,0.0
4,2015,1,1,4,DL,2324,N3751B,SLC,ATL,40,...,0.0,0.0,0.0,0.0,2015-01-01,1,1,SLC,ATL,0.0


2. KIỂU DỮ LIỆU


,Data_Type,Non_Null,Unique_Values
YEAR,int64,875881,1
MONTH,int64,875881,12
DAY,int64,875881,31
DAY_OF_WEEK,int64,875881,7
AIRLINE,str,875881,1
FLIGHT_NUMBER,int64,875881,2449
TAIL_NUMBER,str,875881,829
ORIGIN_AIRPORT,str,875881,157
DESTINATION_AIRPORT,str,875881,157
SCHEDULED_DEPARTURE,int64,875881,1156


3. MISSING VALUES


...

Tỷ lệ phần trăm từng nhóm:


,Percentage
IS_DELAYED,
0.0,85.89
1.0,13.47
NaN,0.64


Số chuyến bị trễ: 118,023
Số chuyến không bị trễ: 752,252
5. THỐNG KÊ CƠ BẢN


,count,mean,std,min,25%,50%,75%,max
YEAR,875881.0,2015.000000,0.000000,2015.0,2015.0,2015.0,2015.0,2015.0
MONTH,875881.0,6.615052,3.361124,1.0,4.0,7.0,9.0,12.0
DAY,875881.0,15.708578,8.762232,1.0,8.0,16.0,23.0,31.0
DAY_OF_WEEK,875881.0,3.908314,1.989460,1.0,2.0,4.0,6.0,7.0
FLIGHT_NUMBER,875881.0,1616.302924,670.719466,2.0,1140.0,1650.0,2169.0,2853.0
SCHEDULED_DEPARTURE,875881.0,1331.822617,484.734464,1.0,916.0,1323.0,1730.0,2359.0
DEPARTURE_TIME,872177.0,1334.800463,495.694798,1.0,921.0,1327.0,1735.0,2400.0
DEPARTURE_DELAY,872177.0,7.369254,36.337405,-61.0,-4.0,-1.0,4.0,1289.0
TAXI_OUT,872094.0,17.608081,9.140596,1.0,12.0,15.0,20.0,180.0
WHEELS_OFF,872094.0,1360.411550,498.023645,1.0,937.0,1341.0,1751.0,2400.0


6. DỮ LIỆU TRÙNG LẶP
Số dòng trùng lặp: 0
Không phát hiện dòng trùng lặp hoàn toàn.
7. TỔNG KẾT
- Số dòng: 875,881
- Số cột: 37
- Tổng giá trị thiếu: 68,331
- Số dòng trùng lặp: 0
- Tỷ lệ chuyến bị trễ: 13.47%
Hoàn thành kiểm tra dữ liệu Delta.
